# DBSCAN Clustering on Well Log Data
## Machine learning for energy applications — Module 3, Exercise 2

In this exercise we apply **DBSCAN** (Density-Based Spatial Clustering of Applications
with Noise) to well log data. Unlike K-Means, DBSCAN does not require specifying the
number of clusters in advance and can discover clusters of arbitrary shape. It also labels
samples in sparse regions as **noise** (label −1). Noise means "not in a dense region at the
chosen `eps` and `min_samples`", not "bad data": noise points are candidates to inspect
(thin beds, fluid contacts, rare lithologies, tool effects) before anyone calls them anomalies.

---
## 0 — Imports and Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN, KMeans
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score

# Plotting defaults
plt.rcParams.update({
    "figure.figsize": (8, 5),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 12,
})

RANDOM_STATE = 42
print("Setup complete.")

---
## 1 — Load and Explore the Data

We load two datasets:
- **log.csv** — a two-feature dataset with SP (spontaneous potential) and GR (gamma ray)
- **w5.csv** — a multi-feature well log dataset

We will primarily work with `log.csv` for 2D visualization, and then apply DBSCAN
to the higher-dimensional `w5.csv`.

In [ ]:
# Load the 2-feature log dataset
df_log = pd.read_csv("log.csv")
print(f"log.csv — Shape: {df_log.shape}")
print(f"Columns: {list(df_log.columns)}")
df_log.head(10)

In [ ]:
# Load the multi-feature well log dataset
df_w5 = pd.read_csv("w5.csv")
print(f"w5.csv — Shape: {df_w5.shape}")
print(f"Columns: {list(df_w5.columns)}")
df_w5.head(10)

In [ ]:
print("=== log.csv summary ===")
print(df_log.describe())
print(f"\nMissing values:\n{df_log.isnull().sum()}")

print("\n=== w5.csv summary ===")
print(df_w5.describe())
print(f"\nMissing values:\n{df_w5.isnull().sum()}")

In [ ]:
# Scatter plot of the 2D log data
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(df_log["SP"], df_log["GR"], alpha=0.4, s=15, color="steelblue")
ax.set_xlabel("SP (Spontaneous Potential)")
ax.set_ylabel("GR (Gamma Ray)")
ax.set_title("Raw Data — SP vs GR")
plt.tight_layout()
plt.show()

---
## 2 — Standardize Features

DBSCAN relies on distance calculations, so features must be on a comparable scale.
We standardize both datasets.

> **Exploratory analysis of fixed datasets.** Each scaler is fitted on all samples of its
> dataset because we only describe these data: nothing is predicted for new samples, so there
> is no training/validation/test split (scikit-learn's DBSCAN cannot label new samples without
> refitting anyway). `eps` is measured in these standardized units.

In [ ]:
# Standardize the 2D log data
X_log = df_log[["SP", "GR"]].dropna().values
scaler_log = StandardScaler()
X_log_scaled = scaler_log.fit_transform(X_log)

print(f"log.csv — Samples: {X_log_scaled.shape[0]}")
print(f"Scaled means: {X_log_scaled.mean(axis=0).round(4)}")
print(f"Scaled stds:  {X_log_scaled.std(axis=0).round(4)}")

# Standardize the multi-feature w5 data
w5_features = ["Pressure", "AI", "VpVs", "SW", "phid", "vcl"]
X_w5 = df_w5[w5_features].dropna().copy()
scaler_w5 = StandardScaler()
X_w5_scaled = scaler_w5.fit_transform(X_w5)

print(f"\nw5.csv — Samples: {X_w5_scaled.shape[0]}")
print(f"Scaled means: {X_w5_scaled.mean(axis=0).round(4)}")
print(f"Scaled stds:  {X_w5_scaled.std(axis=0).round(4)}")

---
## 3 — Apply DBSCAN with Default Parameters

DBSCAN has two key parameters:
- **eps** — the maximum distance between two samples for them to be considered neighbours
- **min_samples** — the minimum number of points required to form a dense region (core point)

We start with `eps=0.5` and `min_samples=5` on the 2D log data.

In [ ]:
dbscan_default = DBSCAN(eps=0.5, min_samples=5)
labels_default = dbscan_default.fit_predict(X_log_scaled)

n_clusters = len(set(labels_default)) - (1 if -1 in labels_default else 0)
n_noise = np.sum(labels_default == -1)

print(f"DBSCAN (eps=0.5, min_samples=5)")
print(f"  Clusters found: {n_clusters}")
print(f"  Noise points:   {n_noise} ({100*n_noise/len(labels_default):.1f}%)")
print(f"\nCluster sizes:")
for c in sorted(set(labels_default)):
    label_name = "Noise" if c == -1 else f"Cluster {c}"
    print(f"  {label_name}: {np.sum(labels_default == c)} samples")

---
## 4 — Visualize Clusters and Noise Points

One of DBSCAN's key advantages is its ability to label points as noise. We colour
cluster members and mark noise points separately.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: original data
axes[0].scatter(X_log_scaled[:, 0], X_log_scaled[:, 1], alpha=0.4, s=15, color="steelblue")
axes[0].set_title("Original Data (Standardized)")
axes[0].set_xlabel("SP (scaled)")
axes[0].set_ylabel("GR (scaled)")

# Right: DBSCAN results
unique_labels = sorted(set(labels_default))
colors = plt.cm.tab10(np.linspace(0, 1, len(unique_labels)))

for label, color in zip(unique_labels, colors):
    mask = labels_default == label
    if label == -1:
        axes[1].scatter(
            X_log_scaled[mask, 0], X_log_scaled[mask, 1],
            c="black", marker="x", s=30, alpha=0.6, label="Noise"
        )
    else:
        axes[1].scatter(
            X_log_scaled[mask, 0], X_log_scaled[mask, 1],
            c=[color], alpha=0.5, s=15, label=f"Cluster {label}"
        )

axes[1].set_title(f"DBSCAN (eps=0.5, min_samples=5) — {n_clusters} clusters")
axes[1].set_xlabel("SP (scaled)")
axes[1].set_ylabel("GR (scaled)")
axes[1].legend(loc="best", fontsize=9)

plt.tight_layout()
plt.show()

---
## 5 — K-Distance Plot to Estimate eps

The **K-distance plot** helps choose `eps`. For each point, we compute the distance to its
k-th nearest neighbour, with k = min_samples and the point itself counted as its first
neighbour (as DBSCAN's `min_samples` counts it). We sort these distances and look for the
"knee" where the curve starts to rise steeply: an `eps` near the knee keeps most samples
as core points and leaves the sparse tail as noise.

In [ ]:
k = 5  # same as min_samples
nn = NearestNeighbors(n_neighbors=k)
nn.fit(X_log_scaled)
distances, indices = nn.kneighbors(X_log_scaled)

# Sort the k-th nearest neighbour distances
k_distances = np.sort(distances[:, k-1])

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(range(len(k_distances)), k_distances, color="steelblue", linewidth=1.5)
ax.set_xlabel("Points (sorted by distance)")
ax.set_ylabel(f"{k}-th Nearest Neighbour Distance")
ax.set_title(f"K-Distance Plot (k={k})")
ax.axhline(y=0.5, color="tomato", linestyle="--", alpha=0.7, label="eps=0.5")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Distance statistics:")
print(f"  Min:    {k_distances.min():.4f}")
print(f"  Median: {np.median(k_distances):.4f}")
print(f"  90th percentile: {np.percentile(k_distances, 90):.4f}")
print(f"  Mean:   {k_distances.mean():.4f}")
print(f"  Max:    {k_distances.max():.4f}")
print(f"Share of samples with {k}-distance <= 0.5: {np.mean(k_distances <= 0.5):.1%}")

---
## 6 — Experiment with Different eps and min_samples

We systematically vary `eps` and `min_samples` to observe how the clustering
results change. This helps build intuition for parameter tuning. The summary table reports
the number of clusters, the noise and the largest cluster; its silhouette is computed on the
clustered points only, because noise (−1) is not a cluster, so read it together with the
noise fraction.

In [ ]:
# Vary eps with fixed min_samples=5
eps_values = [0.2, 0.3, 0.5, 0.8, 1.0, 1.5]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for ax, eps in zip(axes.ravel(), eps_values):
    db = DBSCAN(eps=eps, min_samples=5)
    labs = db.fit_predict(X_log_scaled)
    n_clust = len(set(labs)) - (1 if -1 in labs else 0)
    n_noi = np.sum(labs == -1)

    for label in sorted(set(labs)):
        mask = labs == label
        if label == -1:
            ax.scatter(X_log_scaled[mask, 0], X_log_scaled[mask, 1],
                       c="black", marker="x", s=20, alpha=0.5)
        else:
            ax.scatter(X_log_scaled[mask, 0], X_log_scaled[mask, 1],
                       alpha=0.4, s=10)

    ax.set_title(f"eps={eps}, clusters={n_clust}, noise={n_noi}", fontsize=10)
    ax.set_xlabel("SP")
    ax.set_ylabel("GR")

fig.suptitle("DBSCAN — Varying eps (min_samples=5)", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Vary min_samples with a fixed eps (chosen from K-distance plot)
min_samples_values = [3, 5, 10, 15, 20, 30]
eps_fixed = 0.5

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for ax, ms in zip(axes.ravel(), min_samples_values):
    db = DBSCAN(eps=eps_fixed, min_samples=ms)
    labs = db.fit_predict(X_log_scaled)
    n_clust = len(set(labs)) - (1 if -1 in labs else 0)
    n_noi = np.sum(labs == -1)

    for label in sorted(set(labs)):
        mask = labs == label
        if label == -1:
            ax.scatter(X_log_scaled[mask, 0], X_log_scaled[mask, 1],
                       c="black", marker="x", s=20, alpha=0.5)
        else:
            ax.scatter(X_log_scaled[mask, 0], X_log_scaled[mask, 1],
                       alpha=0.4, s=10)

    ax.set_title(f"min_samples={ms}, clusters={n_clust}, noise={n_noi}", fontsize=10)
    ax.set_xlabel("SP")
    ax.set_ylabel("GR")

fig.suptitle(f"DBSCAN — Varying min_samples (eps={eps_fixed})", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Summary table of parameter combinations
# Noise (-1) is not a cluster: the silhouette is computed on the clustered points only,
# needs at least two clusters, and is read together with the noise fraction.
results = []
for eps in [0.3, 0.5, 0.8, 1.0]:
    for ms in [3, 5, 10, 15]:
        db = DBSCAN(eps=eps, min_samples=ms)
        labs = db.fit_predict(X_log_scaled)
        clustered = labs != -1
        n_clust = len(set(labs[clustered]))
        n_noi = int(np.sum(~clustered))
        largest = int(np.bincount(labs[clustered]).max()) if n_clust > 0 else 0
        sil = (silhouette_score(X_log_scaled[clustered], labs[clustered])
               if n_clust > 1 else np.nan)
        results.append({
            "eps": eps, "min_samples": ms,
            "n_clusters": n_clust, "n_noise": n_noi,
            "noise_%": round(100 * n_noi / len(labs), 1),
            "largest_cluster": largest,
            "silhouette_no_noise": round(sil, 4),
        })

df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))

---
## 7 — Compare DBSCAN with K-Means on the Same Data

DBSCAN and K-Means make fundamentally different assumptions. K-Means assumes
compact, roughly spherical clusters of similar size. DBSCAN finds clusters based on density
and can detect arbitrarily shaped clusters and noise.

We keep `eps = 0.5`, `min_samples = 5`: 0.5 lies at the start of the bend in the k-distance
plot (Section 5), and the eps sweep in Section 6 (min_samples = 5) gives one cluster for every
eps from 0.5 to 1.5, so the conclusion does not hinge on the exact value. DBSCAN finds a single
cluster here, and K-Means needs at least two, so we fit K = 2 and discuss what each method is
saying. The DBSCAN silhouette is undefined for a single cluster.

In [ ]:
# DBSCAN configuration read from the k-distance plot and checked with the sweep above
chosen_eps = 0.5
chosen_ms = 5
dbscan_best = DBSCAN(eps=chosen_eps, min_samples=chosen_ms)
labels_dbscan = dbscan_best.fit_predict(X_log_scaled)
n_clusters_dbscan = len(set(labels_dbscan)) - (1 if -1 in labels_dbscan else 0)

# Fit K-Means with the same number of clusters (K-Means needs K >= 2)
k_kmeans = max(n_clusters_dbscan, 2)
kmeans = KMeans(n_clusters=k_kmeans, random_state=RANDOM_STATE, n_init=10)
labels_kmeans = kmeans.fit_predict(X_log_scaled)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# DBSCAN plot
for label in sorted(set(labels_dbscan)):
    mask = labels_dbscan == label
    if label == -1:
        axes[0].scatter(X_log_scaled[mask, 0], X_log_scaled[mask, 1],
                        c="black", marker="x", s=30, alpha=0.6, label="Noise")
    else:
        axes[0].scatter(X_log_scaled[mask, 0], X_log_scaled[mask, 1],
                        alpha=0.4, s=15, label=f"Cluster {label}")

axes[0].set_title(f"DBSCAN (eps={chosen_eps}, min_samples={chosen_ms})")
axes[0].set_xlabel("SP (scaled)")
axes[0].set_ylabel("GR (scaled)")
axes[0].legend(fontsize=8)

# K-Means plot
scatter = axes[1].scatter(
    X_log_scaled[:, 0], X_log_scaled[:, 1],
    c=labels_kmeans, cmap="tab10", alpha=0.4, s=15
)
# Plot centroids
axes[1].scatter(
    kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1],
    c="red", marker="*", s=200, edgecolors="black", linewidths=1,
    label="Centroids"
)
axes[1].set_title(f"K-Means (K={k_kmeans})")
axes[1].set_xlabel("SP (scaled)")
axes[1].set_ylabel("GR (scaled)")
axes[1].legend(fontsize=8)

fig.suptitle("DBSCAN vs K-Means Comparison", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# Quantitative comparison. The DBSCAN silhouette uses the clustered points only
# (noise is not a cluster) and is undefined when there is a single cluster.
clustered = labels_dbscan != -1
sil_dbscan = (silhouette_score(X_log_scaled[clustered], labels_dbscan[clustered])
              if n_clusters_dbscan > 1 else float("nan"))
sil_kmeans = silhouette_score(X_log_scaled, labels_kmeans)

sil_text = f"{sil_dbscan:.4f}" if n_clusters_dbscan > 1 else "undefined (one cluster)"
print(f"DBSCAN — Clusters: {n_clusters_dbscan}, Noise: {np.sum(~clustered)}, "
      f"Silhouette (noise excluded): {sil_text}")
print(f"K-Means — Clusters: {k_kmeans}, Noise: 0, "
      f"Silhouette: {sil_kmeans:.4f}")

---
## 8 — Apply DBSCAN to Multi-Feature Well Log Data (w5.csv)

Now we apply DBSCAN to the six standardized W5 logs. Distances grow with the number of
features, so we read `eps` again from the k-distance plot of these data instead of assuming
that the log.csv value transfers, and then check how the result changes with `eps`.

In [ ]:
# K-distance plot for the w5 data (k = min_samples = 5, the point itself counted)
k_w5 = 5
nn_w5 = NearestNeighbors(n_neighbors=k_w5)
nn_w5.fit(X_w5_scaled)
dist_w5, _ = nn_w5.kneighbors(X_w5_scaled)
k_dist_w5 = np.sort(dist_w5[:, k_w5-1])

# eps read from this plot: the curve starts to rise steeply at about 0.5
# (close to the 90th percentile of the 5-distances); the steep tail is sparse.
eps_w5 = 0.5

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(range(len(k_dist_w5)), k_dist_w5, color="steelblue", linewidth=1.5)
ax.axhline(y=eps_w5, color="tomato", linestyle="--", alpha=0.7, label=f"chosen eps={eps_w5}")
ax.set_xlabel("Points (sorted by distance)")
ax.set_ylabel(f"{k_w5}-th Nearest Neighbour Distance")
ax.set_title(f"K-Distance Plot — w5.csv (k={k_w5})")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Median {k_w5}-distance: {np.median(k_dist_w5):.2f}; "
      f"90th percentile: {np.percentile(k_dist_w5, 90):.2f}")
print(f"Share of samples with {k_w5}-distance <= eps: {np.mean(k_dist_w5 <= eps_w5):.1%}")

In [ ]:
# Apply DBSCAN to w5 data with the eps chosen from the k-distance plot
dbscan_w5 = DBSCAN(eps=eps_w5, min_samples=5)
labels_w5 = dbscan_w5.fit_predict(X_w5_scaled)

n_clusters_w5 = len(set(labels_w5)) - (1 if -1 in labels_w5 else 0)
n_noise_w5 = np.sum(labels_w5 == -1)

print(f"DBSCAN on w5.csv (eps={eps_w5}, min_samples=5)")
print(f"  Clusters found: {n_clusters_w5}")
print(f"  Noise points:   {n_noise_w5} ({100*n_noise_w5/len(labels_w5):.1f}%)")
largest_w5 = np.bincount(labels_w5[labels_w5 != -1]).max()
print(f"  Largest cluster: {largest_w5} samples ({100*largest_w5/len(labels_w5):.0f}%)")

# Sensitivity: how do clusters, noise and the largest cluster change with eps?
rows = []
for eps in [0.3, 0.5, 0.8, 1.0]:
    labs = DBSCAN(eps=eps, min_samples=5).fit_predict(X_w5_scaled)
    clustered = labs != -1
    rows.append({"eps": eps, "n_clusters": len(set(labs[clustered])),
                 "noise_%": round(100 * np.mean(~clustered), 1),
                 "largest_cluster": int(np.bincount(labs[clustered]).max())})
print(pd.DataFrame(rows).to_string(index=False))

# Visualize in AI vs VpVs space
fig, ax = plt.subplots(figsize=(8, 6))
for label in sorted(set(labels_w5)):
    mask = labels_w5 == label
    if label == -1:
        ax.scatter(X_w5["AI"].values[mask], X_w5["VpVs"].values[mask],
                   c="black", marker="x", s=30, alpha=0.6, label="Noise")
    else:
        ax.scatter(X_w5["AI"].values[mask], X_w5["VpVs"].values[mask],
                   alpha=0.5, s=15, label=f"Cluster {label}")

ax.set_xlabel("AI")
ax.set_ylabel("VpVs")
ax.set_title(f"DBSCAN on w5.csv — AI vs VpVs (eps={eps_w5})")
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

---
## 9 — Analyse Noise Points Detected by DBSCAN

Points labelled as noise (cluster = -1) lie in low-density regions at the chosen `eps` and
`min_samples`; that is not the same as bad data. Compare them with the other samples in
original units and check where they occur along depth before interpreting them: measurement
problems, thin beds, unusual lithologies and fluid contacts are all candidates.

In [ ]:
# Analyse noise points from the 2D log data (original units)
noise_mask_log = labels_dbscan == -1
normal_mask_log = labels_dbscan != -1

print(f"Noise points: {noise_mask_log.sum()} / {len(labels_dbscan)}")
print(f"\n--- Noise points statistics (original units) ---")
noise_df = pd.DataFrame(X_log[noise_mask_log], columns=["SP", "GR"])
print(noise_df.describe())

print(f"\n--- Normal points statistics (original units) ---")
normal_df = pd.DataFrame(X_log[normal_mask_log], columns=["SP", "GR"])
print(normal_df.describe())

In [ ]:
# Visualize outlier distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter with noise highlighted
axes[0].scatter(X_log_scaled[normal_mask_log, 0], X_log_scaled[normal_mask_log, 1],
                alpha=0.3, s=10, color="steelblue", label="Normal")
axes[0].scatter(X_log_scaled[noise_mask_log, 0], X_log_scaled[noise_mask_log, 1],
                c="red", marker="o", s=40, alpha=0.8, label="Noise / Outlier")
axes[0].set_xlabel("SP (scaled)")
axes[0].set_ylabel("GR (scaled)")
axes[0].set_title("Noise Points Highlighted")
axes[0].legend()

# Distribution comparison (SP)
axes[1].hist(X_log_scaled[normal_mask_log, 0], bins=30, alpha=0.5, color="steelblue",
             label="Normal SP", density=True)
axes[1].hist(X_log_scaled[noise_mask_log, 0], bins=15, alpha=0.7, color="red",
             label="Noise SP", density=True)
axes[1].set_xlabel("SP (scaled)")
axes[1].set_ylabel("Density")
axes[1].set_title("SP Distribution: Normal vs Noise")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Noise analysis on w5 data (original units)
noise_mask_w5 = labels_w5 == -1

if noise_mask_w5.sum() > 0:
    print(f"w5.csv — Noise points: {noise_mask_w5.sum()} / {len(labels_w5)}")
    print(f"\n--- Noise feature means vs the other (clustered) samples ---")
    comparison = pd.DataFrame({
        "Normal Mean": X_w5[~noise_mask_w5].mean(),
        "Outlier Mean": X_w5[noise_mask_w5].mean(),
        "Difference": X_w5[noise_mask_w5].mean() - X_w5[~noise_mask_w5].mean()
    })
    print(comparison.round(4).to_string())

    # What do the noise points have in common? SW < 1: not fully water-saturated
    sw_below_1 = (X_w5["SW"] < 1).to_numpy()
    print(f"\nShare with SW < 1: noise {sw_below_1[noise_mask_w5].mean():.0%}, "
          f"other samples {sw_below_1[~noise_mask_w5].mean():.0%}")

    # Thick intervals or thin runs along depth? (w5.csv is sorted by depth)
    noise_pos = np.flatnonzero(noise_mask_w5)
    runs = np.split(noise_pos, np.flatnonzero(np.diff(noise_pos) != 1) + 1)
    print(f"Noise points form {len(runs)} runs along depth; "
          f"median run length {np.median([len(r) for r in runs]):.0f} samples")
else:
    print("No noise points detected in w5.csv with current parameters.")
    print("Try decreasing eps or increasing min_samples.")

---
## Summary

In this exercise we covered:

1. **Data Loading** — working with both 2D (log.csv) and multi-dimensional (w5.csv) well log data.
2. **Feature Standardization** — essential preprocessing for distance-based algorithms (exploratory: fitted on each fixed dataset).
3. **DBSCAN with Default Parameters** — applying DBSCAN and understanding its output (clusters + noise).
4. **Cluster Visualization** — plotting clusters and clearly marking noise points.
5. **K-Distance Plot** — reading `eps` from the knee of the sorted k-distances.
6. **Parameter Sensitivity** — tabulating clusters, noise and the largest cluster as eps and min_samples change.
7. **DBSCAN vs K-Means** — understanding the trade-offs between density-based and centroid-based clustering.
8. **Noise Analysis** — comparing noise points with the other samples, in original units and along depth, before calling them anomalies.

DBSCAN is particularly valuable in petrophysics when:
- The number of clusters is unknown
- Clusters have irregular shapes
- Low-density samples should be flagged for review (e.g., bad hole, thin beds, unusual formations)